In [34]:
from sage.all import *
import random
from itertools import combinations
import os


def random_poly(
    R,
    p_lin=0.50,
    p_quad=0.50,
    p_higher=0.70,
    p_const=0.5,
    max_degree=None
):
    """Random polynomial over GF(2) with denser, higher-degree monomials."""
    x = R.gens()
    n = len(x)
    
    if max_degree is None:
        max_degree = n
        
    terms = []

    # Linear terms
    for i in range(n):
        if random.random() < p_lin:
            terms.append(x[i])

    # Quadratic terms
    if max_degree >= 2:
        for i, j in combinations(range(n), 2):
            if random.random() < p_quad:
                terms.append(x[i] * x[j])

    # Cubic and higher-degree terms
    for degree in range(3, min(max_degree, n) + 1):
        for indices in combinations(range(n), degree):
            if random.random() < p_higher:
                monomial = R.one()
                for i in indices:
                    monomial *= x[i]
                terms.append(monomial)

    # Constant term
    if random.random() < p_const:
        terms.append(R.one())

    return sum(terms, R.zero())


def find_gb_system(R, n, max_attempts=None):
    """Generate n random polynomials until the reduced Groebner basis 
    has size n AND its max degree is >= 2."""
    attempt = 0
    while max_attempts is None or attempt < max_attempts:
        attempt += 1
        eqs = [random_poly(R, max_degree=n) for _ in range(n)]
        gb = R.ideal(eqs).groebner_basis()
        
        if len(gb) == n:
            max_deg = max((p.degree() for p in gb), default=0)
            if max_deg >= 2:
                return list(gb)
                
    raise RuntimeError(f"No system with GB of size {n} and degree >= 2 found.")


def generate_and_save_systems(n, m, folder_path="gf2_gb_systems", max_attempts=1000, seed=None):
    """Generates m Gröbner basis systems (degree >= 2) and saves them safely."""
    if seed is not None:
        random.seed(seed)
        set_random_seed(seed)

    os.makedirs(folder_path, exist_ok=True)
    R = PolynomialRing(GF(2), n, [f"x{i}" for i in range(n)], order="degrevlex")

    print(f"Generating and saving {m} GB systems (n={n}, degree >= 2)...")
    for i in range(m):
        sys_gb = find_gb_system(R, n, max_attempts)
        str_gb = [str(p) for p in sys_gb]
        filepath = os.path.join(folder_path, f"system_{i}_n{n}.sobj")
        save(str_gb, filepath)
        print(f"  -> System {i + 1}/{m} saved. (Max degree: {max(p.degree() for p in sys_gb)})")
    
    print("All systems successfully generated and saved!")

# --- RUN CELL 1 ---
# generate_and_save_systems(n=8, m=10, folder_path="gf2_gb_systems", seed=42)

In [117]:
generate_and_save_systems(n=4, m=100, folder_path="my_gb_systems")

Generating and saving 100 GB systems (n=4, degree >= 2)...
  -> System 1/100 saved. (Max degree: 2)
  -> System 2/100 saved. (Max degree: 2)
  -> System 3/100 saved. (Max degree: 2)
  -> System 4/100 saved. (Max degree: 2)
  -> System 5/100 saved. (Max degree: 2)
  -> System 6/100 saved. (Max degree: 2)
  -> System 7/100 saved. (Max degree: 2)
  -> System 8/100 saved. (Max degree: 3)
  -> System 9/100 saved. (Max degree: 2)
  -> System 10/100 saved. (Max degree: 2)
  -> System 11/100 saved. (Max degree: 2)
  -> System 12/100 saved. (Max degree: 2)
  -> System 13/100 saved. (Max degree: 2)
  -> System 14/100 saved. (Max degree: 2)
  -> System 15/100 saved. (Max degree: 2)
  -> System 16/100 saved. (Max degree: 2)
  -> System 17/100 saved. (Max degree: 2)
  -> System 18/100 saved. (Max degree: 2)
  -> System 19/100 saved. (Max degree: 2)
  -> System 20/100 saved. (Max degree: 2)
  -> System 21/100 saved. (Max degree: 2)
  -> System 22/100 saved. (Max degree: 2)
  -> System 23/100 saved. 

In [125]:

from sage.all import *
import random
import os
import glob


# ============================================================
# 1. BOOLEAN REDUCTION
# ============================================================

def boolean_reduce(f, R):
    """
    Reduce a polynomial over GF(2) using x_i^k = x_i
    for every k >= 1.

    Coefficients are already computed modulo 2 in GF(2).
    """
    x = R.gens()
    result = R.zero()

    for exponents, coefficient in f.dict().items():
        if coefficient == 0:
            continue

        # Replace every positive exponent with 1.
        monomial = R.one()

        for i, exponent in enumerate(exponents):
            if exponent > 0:
                monomial *= x[i]

        result += coefficient * monomial

    return result


def reduce_system(system, R):
    """Apply Boolean reduction to every polynomial."""
    return [boolean_reduce(f, R) for f in system]


# ============================================================
# 2. LOAD SAVED SYSTEMS
# ============================================================

def load_systems(n, folder_path="gf2_gb_systems"):
    """
    Load saved polynomial systems for n variables.

    Each file contains a list of polynomial strings,
    as saved by the original generator.
    """
    R = PolynomialRing(
        GF(2),
        n,
        [f"x{i}" for i in range(n)],
        order="degrevlex"
    )

    pattern = os.path.join(
        folder_path, f"system_*_n{n}.sobj"
    )
    files = sorted(glob.glob(pattern))

    systems = []

    for filepath in files:
        saved_polynomials = load(filepath)

        # Reconstruct polynomial objects in R.
        system = [R(str(p)) for p in saved_polynomials]

        if len(system) != n:
            print(f"Skipping invalid system: {filepath}")
            continue

        systems.append({
            "file": filepath,
            "polynomials": system
        })

    print(f"Loaded {len(systems)} systems for n={n}.")

    return R, systems


# ============================================================
# 3. RANDOM SHUFFLE
# ============================================================

def shuffle_system(system, R):
    """
    Randomly permute the input variables of a system.

    Example:
        x0 -> x2
        x1 -> x0
        x2 -> x1

    The same permutation is applied consistently
    to every polynomial in the system.
    """
    x = R.gens()
    n = len(x)

    permutation = list(range(n))
    random.shuffle(permutation)

    shuffled = [
        f(*[x[permutation[i]] for i in range(n)])
        for f in system
    ]

    return reduce_system(shuffled, R), permutation


# ============================================================
# 4. COMPOSITION
# ============================================================


def compose_systems(F, G, R, shuffle=True):
    """
    Reverse composition order:
        result[i] = g_i(f0, f1, ..., f[n-1])

    F is the existing composed system.
    G is the newly selected system.

    Optionally shuffle the variables of F before
    substituting F into G.
    """
    n = len(R.gens())

    if len(F) != n or len(G) != n:
        raise ValueError(
            "Both systems must contain exactly n polynomials."
        )

    if shuffle:
        F, permutation = shuffle_system(F, R)
    else:
        permutation = list(range(n))

    # Reverse the composition order: G(F(x)).
    composed = [
        g(*F)
        for g in G
    ]

    # Reduce after every composition.
    composed = reduce_system(composed, R)

    return composed, permutation


# ============================================================
# 5. COMPOSE s RANDOMLY SELECTED SYSTEMS
# ============================================================

def generate_composed_system(
    n,
    s,
    folder_path="gf2_gb_systems",
    seed=None
):
    """
    Randomly select s saved systems and compose them
    sequentially, shuffling variables at each step.

    Returns:
        R          : polynomial ring
        final      : final composed polynomial system
        selected   : selected source filenames
        shuffles   : variable permutations used
    """
    if seed is not None:
        random.seed(seed)
        set_random_seed(seed)

    R, systems = load_systems(n, folder_path)

    if s < 1:
        raise ValueError("s must be at least 1.")

    if s > len(systems):
        raise ValueError(
            f"Requested {s} systems, but only "
            f"{len(systems)} valid systems were found."
        )

    # Select distinct saved systems randomly.
    selected_systems = random.sample(systems, s)

    selected_files = [
        item["file"] for item in selected_systems
    ]

    # Start with the first selected system.
    final = reduce_system(
        selected_systems[0]["polynomials"], R
    )

    shuffles = []

    print("\nSelected systems:")
    for filepath in selected_files:
        print(" ", filepath)

    # Sequential composition.
    for step in range(1, s):
        next_system = selected_systems[step]["polynomials"]

        final, permutation = compose_systems(
            final,
            next_system,
            R,
            shuffle=True
        )

        shuffles.append(permutation)

        print(
            f"Composition {step}/{s - 1} completed. "
            f"Shuffle: {permutation}"
        )

    print("\nFinal composed system:")
    for i, f in enumerate(final):
        print(f"f{i} = {f}")

    print("\nFinal polynomial degrees:")
    for i, f in enumerate(final):
        print(f"deg(f{i}) = {f.degree()}")

    return R, final, selected_files, shuffles




In [146]:
# ============================================================
# 6. EXAMPLE
# ============================================================

R, final_system, selected_files, shuffles = (
    generate_composed_system(
        n=4,
        s=5,
        folder_path="my_gb_systems",
    )
)

Loaded 100 systems for n=4.

Selected systems:
  my_gb_systems/system_17_n4.sobj
  my_gb_systems/system_33_n4.sobj
  my_gb_systems/system_30_n4.sobj
  my_gb_systems/system_83_n4.sobj
  my_gb_systems/system_39_n4.sobj
Composition 1/4 completed. Shuffle: [0, 1, 2, 3]
Composition 2/4 completed. Shuffle: [2, 1, 0, 3]
Composition 3/4 completed. Shuffle: [2, 1, 0, 3]
Composition 4/4 completed. Shuffle: [0, 3, 2, 1]

Final composed system:
f0 = x0 + x1 + x3
f1 = x0*x1*x3 + x0*x2*x3 + x1*x2*x3 + x0*x2 + x1*x2 + x0*x3 + x1*x3 + x3
f2 = x1*x2*x3 + x1*x2 + x2*x3 + x0 + x2
f3 = x1*x2 + x1 + x2 + x3

Final polynomial degrees:
deg(f0) = 1
deg(f1) = 3
deg(f2) = 3
deg(f3) = 2


In [151]:

from sage.all import *
import random as py_random
import os
import glob


# ============================================================
# 1. BOOLEAN REDUCTION
# ============================================================

def boolean_reduce(f, R):
    """Reduce x_i^k to x_i for every k >= 1 over GF(2)."""
    result = R.zero()
    exponents_dict = f.dict()

    for exponents, coefficient in exponents_dict.items():
        if coefficient == 0:
            continue

        monomial = R.one()

        for i, exponent in enumerate(exponents):
            if exponent > 0:
                monomial *= R.gen(i)

        result += coefficient * monomial

    return result


def reduce_system(system, R):
    return [boolean_reduce(f, R) for f in system]


# ============================================================
# 2. LOAD SAVED SYSTEMS
# ============================================================

def load_systems(n, folder_path):
    R = PolynomialRing(
        GF(2), n,
        [f"x{i}" for i in range(n)],
        order="degrevlex"
    )

    pattern = os.path.join(folder_path, f"system_*_n{n}.sobj")
    files = sorted(glob.glob(pattern))
    systems = []

    for filepath in files:
        try:
            saved_polynomials = load(filepath)
            system = [R(str(p)) for p in saved_polynomials]

            if len(system) == n:
                systems.append({
                    "file": filepath,
                    "polynomials": system
                })
        except Exception as e:
            print(f"Skipping {filepath}: {e}")

    print(f"Loaded {len(systems)} systems from '{folder_path}'.")

    return R, systems


# ============================================================
# 3. SHUFFLE VARIABLES
# ============================================================

def shuffle_system(system, R):
    """Apply one random variable permutation to the whole system."""
    n = R.ngens()
    x = R.gens()

    permutation = list(range(n))
    py_random.shuffle(permutation)

    shuffled = [
        f(*[x[permutation[i]] for i in range(n)])
        for f in system
    ]

    return reduce_system(shuffled, R), permutation


# ============================================================
# 4. REVERSED COMPOSITION: G(F(x))
# ============================================================

def compose_systems(F, G, R, shuffle=True):
    """
    Compose in reversed order:
        result[i] = G[i](F[0], ..., F[n-1])

    Shuffle F before substitution, if requested.
    Boolean-reduce immediately after composition.
    """
    n = R.ngens()

    if len(F) != n or len(G) != n:
        raise ValueError("Each system must contain exactly n polynomials.")

    if shuffle:
        F, permutation = shuffle_system(F, R)
    else:
        permutation = list(range(n))

    composed = [g(*F) for g in G]
    composed = reduce_system(composed, R)

    return composed, permutation


# ============================================================
# 5. GENERATE ONE COMPOSED SYSTEM
# ============================================================

def generate_composed_system(
    n=4,
    s=5,
    folder_path="my_gb_systems"
):
    R, systems = load_systems(n, folder_path)

    if len(systems) < s:
        raise ValueError(
            f"Need {s} distinct systems, but only "
            f"{len(systems)} valid systems were found in "
            f"'{folder_path}'."
        )

    selected = py_random.sample(systems, s)

    selected_files = [item["file"] for item in selected]
    shuffles = []

    # Start with the first selected system.
    final_system = reduce_system(selected[0]["polynomials"], R)

    # Compose each subsequent system in reversed order.
    for step in range(1, s):
        next_system = selected[step]["polynomials"]

        final_system, permutation = compose_systems(
            final_system,
            next_system,
            R,
            shuffle=True
        )

        shuffles.append(permutation)

    return R, final_system, selected_files, shuffles


# ============================================================
# 6. GENERATE AND SAVE 100 ACCEPTED COMPOSED SYSTEMS
# ============================================================

def generate_and_save_composed_systems(
    n=4,
    s=5,
    target=100,
    folder_path="my_gb_systems",
    output_folder="composed_systems_4",
    max_attempts=10000
):
    os.makedirs(output_folder, exist_ok=True)

    accepted = 0
    attempts = 0

    while accepted < target and attempts < max_attempts:
        attempts += 1

        # Run the complete composition process.
        R, final_system, selected_files, shuffles = (
            generate_composed_system(
                n=n,
                s=s,
                folder_path=folder_path
            )
        )

        # Count distinct monomials in every final polynomial.
        term_counts = [len(f.dict()) for f in final_system]

        # Accept only if every polynomial has >= 3 terms.
        if all(count >= 3 for count in term_counts):
            accepted += 1

            filepath = os.path.join(
                output_folder,
                f"composed_system_{accepted}_n{n}.sobj"
            )

            # Save the final composed polynomials as strings.
            save([str(f) for f in final_system], filepath)

            print(
                f"ACCEPTED {accepted}/{target} | "
                f"Attempt {attempts} | "
                f"Term counts: {term_counts}"
            )
        else:
            print(
                f"Rejected attempt {attempts} | "
                f"Term counts: {term_counts}"
            )

    print("\n========== COMPLETE ==========")
    print(f"Accepted systems: {accepted}/{target}")
    print(f"Attempts: {attempts}")
    print(f"Output folder: {output_folder}")

    if accepted < target:
        print("Warning: Target not reached within max_attempts.")


# ============================================================
# 7. RUN
# ============================================================

generate_and_save_composed_systems(
    n=4,
    s=5,
    target=100,
    folder_path="my_gb_systems",
    output_folder="composed_systems_4",
    max_attempts=10000
)

Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 1 | Term counts: [1, 2, 5, 9]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 2 | Term counts: [1, 4, 1, 3]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 3 | Term counts: [2, 0, 2, 4]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 4 | Term counts: [1, 5, 4, 6]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 5 | Term counts: [1, 1, 6, 6]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 6 | Term counts: [1, 2, 6, 2]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 7 | Term counts: [2, 2, 2, 4]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 8 | Term counts: [2, 1, 4, 3]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 9 | Term counts: [6, 0, 6, 4]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 10 | Term counts: [0, 7, 4, 7]
Loaded 100 systems from 'my_gb_systems'.
Rejected attempt 11 | Term counts: [0, 4, 3, 0]
Loaded 100 systems from 'my_gb

In [152]:
import random as py_random
import glob
from sage.all import load

files = glob.glob("composed_systems_4/*.sobj")

if files:
    filepath = py_random.choice(files)
    system = load(filepath)

    print(f"System: {filepath}\n")
    for i, p in enumerate(system):
        print(f"f{i} = {p}")
else:
    print("No composed systems found.")

System: composed_systems_4/composed_system_94_n4.sobj

f0 = x0*x3 + x0 + x3
f1 = x0*x3 + x0 + x3 + 1
f2 = x0*x1 + x0*x3 + x1*x3 + x0 + 1
f3 = x0*x3 + x0 + x3


In [153]:

import os
import glob
import json
from sage.all import load

input_folder = "composed_systems_4"
output_folder = "polynomial systems"

os.makedirs(output_folder, exist_ok=True)

files = glob.glob(os.path.join(input_folder, "*.sobj"))

for filepath in files:
    system = load(filepath)

    data = {"polynomials": [str(p) for p in system]}

    filename = os.path.splitext(os.path.basename(filepath))[0] + ".json"
    output_path = os.path.join(output_folder, filename)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

    print(f"Saved: {output_path}")

print(f"\nConverted {len(files)} systems to JSON.")

Saved: polynomial systems/composed_system_97_n4.json
Saved: polynomial systems/composed_system_87_n4.json
Saved: polynomial systems/composed_system_9_n4.json
Saved: polynomial systems/composed_system_11_n4.json
Saved: polynomial systems/composed_system_100_n4.json
Saved: polynomial systems/composed_system_73_n4.json
Saved: polynomial systems/composed_system_63_n4.json
Saved: polynomial systems/composed_system_55_n4.json
Saved: polynomial systems/composed_system_45_n4.json
Saved: polynomial systems/composed_system_37_n4.json
Saved: polynomial systems/composed_system_27_n4.json
Saved: polynomial systems/composed_system_18_n4.json
Saved: polynomial systems/composed_system_1_n4.json
Saved: polynomial systems/composed_system_44_n4.json
Saved: polynomial systems/composed_system_54_n4.json
Saved: polynomial systems/composed_system_26_n4.json
Saved: polynomial systems/composed_system_36_n4.json
Saved: polynomial systems/composed_system_19_n4.json
Saved: polynomial systems/composed_system_86_n4